In [1]:
import os
import sys
import subprocess
from datetime import datetime
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xskillscore as xs
import netCDF4

In [2]:
BASE_DIR = '/Users/madisonrichardson/netpp/monthly'
BIN_DIR = '/Users/madisonrichardson/netpp/bin'
DATA_DIR = os.path.join(BASE_DIR, 'data', 'monthly')
WORK_DIR = os.path.join(BASE_DIR, 'data', 'work')
RESOURCES_DIR = os.path.join(BASE_DIR, 'resources')
CORRELATIONS_DIR = os.path.join(DATA_DIR, 'correlations')

In [3]:
def validate_params(startyear, endyear, percent_keep):
    """
    Validates the inputted parameters to ensure
    correct ranges and logical order.

    Args:
        startyear (str): The start year in 'YYYYMM' format.
        endyear (str): The end year in 'YYYYMM' format.
        percent_keep (float): Percentage of grid points to keep (0 to 1)

    Raises:
        ValueError: The startyear is not earlier than endyear.
        ValueError: The percent_keep is outside the range (0 to 1).
    """
    if startyear > endyear:
        raise ValueError("The start year must be earlier than the end year.")
    if not (0.0 <= percent_keep <= 1.0):
        raise ValueError("percent_keep must be between 0 and 1")

In [4]:
def parse_dates(startyear, endyear):
    """
    Parses the start and end years in 'YYYYMM' format
    and extracts the corresponding years and months as
    separate integer values.
 
    Args:
        startyear (str): The start date in 'YYYYMM' format.
        endyear (str): The end date in 'YYYYMM' format.

    Returns:
        tuple: A tuple containing four integers:
        (syear, smonth, eyear, emonth).
            - syear (int): The year component of the start date.
            - smonth (int): The month component of the start date.
            - eyear (int): The year component of the end date.
            - emonth (int): The monht component of the end date.
    """
    syear = int(np.floor(int(startyear) / 100))
    smonth = int(startyear) - syear * 100
    eyear = int(np.floor(int(endyear) / 100))
    emonth = int(endyear) - eyear * 100
    return syear, smonth, eyear, emonth

In [5]:
def reshape_data_block(data, block_size):
    """
    Reshapes a 2D array's latitude dimension into
    smaller blocks for efficient processing.

    Args:
        data (np.ndarray): A 2D array with shape (time, latitude, longitude)
        where the latitude dimension (axis=1) will be divide into blocks.
        block_size (int): The number of latitude grid points in each block.

    Returns:
        np.ndarray: A 2D array where the latitude indices are reshaped
        into blocks of the specified size, with shape (num_blocks, block_size).
    """
    ny_index = np.arange(data.shape[1])
    num_blocks = int(data.shape[1] / block_size)
    return np.reshape(ny_index, [num_blocks, block_size])

In [6]:
def xr_open_ds(e_id, e_source='http://localhost:8080/erddap', dap='griddap', var_name=None):
    """
    Open a remote ERDDAP dataset.
    """
    # remove any trailing /
    e_source = e_source.rstrip("/")

    erddap_url = '/'.join([e_source,
                           dap,
                           e_id
                           ])
    
    ds = xr.open_dataset(erddap_url)

    if var_name:
        if var_name in ds:
            return ds[var_name]
        else:
            print(f"Variable '{var_name}' not found in the dataset.")
            print("Available variables:")
            print(list(ds.variables.keys()))
            raise KeyError(f"Variable  '{var_name}' not found.")

    return ds

In [7]:
# Set parameters
startyear = "201801"
endyear = "202212"
source1 = "modis"
source2 = "noaa20"
ncvar = "productivity"
timeseries_type = "data"
percent_keep = 0.5
overwrite = True
ny_block = 20

# For ERDDAP function
erddap_url = "http://localhost:8080/erddap"
modis_id = "productivity_modis_aqua_monthly"
noaa20_id = "productivity_viirs_noaa20_monthly"

# Validate parameters
try:
    validate_params(startyear, endyear, percent_keep)
except ValueError as e:
    print(f"Parameter validation error: {e}")
    sys.exit(1)

In [8]:
# Parse and validate inputs
syear, smonth, eyear, emonth = parse_dates(startyear, endyear)

# Generate monthly date range from the start to the end
time_bgn = np.datetime64('{}-{:02d}'.format(syear, smonth), 'M')
time_end = np.datetime64('{}-{:02d}'.format(eyear, emonth), 'M')
dtM = pd.to_datetime(np.arange(time_bgn, time_end+1, dtype='datetime64[M]'))
ntM = len(dtM)
yy = dtM.year.values
mm = dtM.month.values

print(f"Generated {ntM} monthly time points from {time_bgn} to {time_end}")

Generated 60 monthly time points from 2018-01 to 2022-12


In [9]:
# Verify directories exist
DIR_LIST = [
    BASE_DIR,
    BIN_DIR,
    DATA_DIR,
    WORK_DIR,
    RESOURCES_DIR,
    CORRELATIONS_DIR
]

for dr in DIR_LIST:
    os.makedirs(dr, exist_ok=True)
print(len(DIR_LIST), 'directories validated')

6 directories validated


In [10]:
# Set up templates for the input and output files
ifile_tmpl = 'productivity_month_{}_{}_9km.nc'
ofile_tmpl = '{}_{}_corr_month_{}_{}_{}_{}_{}_{:03d}percent.nc'

# Calculate correlations for ncvar variable wanted
ncvar_list = [ncvar]
for ncvar in ncvar_list:
    # Set up directories and output file parameters
    source_list = np.sort([source1, source2])
    source_comb = '{}_{}'.format(source_list[0], source_list[1])
    odir = os.path.join(
        CORRELATIONS_DIR,
        source_comb
    )
    odir_filelist = os.listdir(odir)

    # Generate output filenaeme
    ofile = ofile_tmpl.format(
        ncvar,
        timeseries_type,
        source_list[0],
        source_list[1],
        '9km',
        startyear,
        endyear,
        int(percent_keep*100)
    )

    # Skip if output file exists and overwrite is False
    if not overwrite and ofile in odir_filelist:
        print(f'{ofile} already exists')
        continue

In [11]:
# Prepare ncgen input and output filenames
now = datetime.now()
ncgen_ofile_nc = f"ncgen_corr_ofile{now:%Y%m%d%H%M%S}.nc"
ncgen_ifile_cdl = 'correlations_{}_month_{}_{}_{}.cdl'.format(
    ncvar,
    source_list[0],
    source_list[1],
    '9km'
)

# Run ncgen command
myCmd1 = ' '.join(['ncgen',
                    '-o',
                    os.path.join(WORK_DIR, ncgen_ofile_nc),
                    os.path.join(RESOURCES_DIR, ncgen_ifile_cdl)
                    ])
print(myCmd1)
print('ncgen', subprocess.call(myCmd1, shell=True))

ncgen -o /Users/madisonrichardson/netpp/monthly/data/work/ncgen_corr_ofile20250102153742.nc /Users/madisonrichardson/netpp/monthly/resources/correlations_productivity_month_modis_noaa20_9km.cdl
ncgen 0


In [12]:
modis_ds = xr_open_ds(
    e_source=erddap_url,
    e_id=modis_id,
    var_name=ncvar
)
print("MODIS Dataset:", modis_ds)

noaa20_ds = xr_open_ds(
    e_source=erddap_url,
    e_id=noaa20_id,
    var_name=ncvar
)
print("NOAA20 Dataset:", noaa20_ds)

MODIS Dataset: <xarray.DataArray 'productivity' (time: 120, latitude: 2160, longitude: 4320)> Size: 4GB
[1119744000 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 960B 2013-01-16T08:00:00 ... 2022-12-16T...
  * latitude   (latitude) float64 17kB 89.96 89.88 89.79 ... -89.88 -89.96
  * longitude  (longitude) float64 35kB -180.0 -179.9 -179.8 ... 179.9 180.0
Attributes:
    colorBarMaximum:        30.0
    colorBarMinimum:        0.0
    coverage_content_type:  modelResult
    ioos_category:          Other
    long_name:              Net Primary Productivity Monthly Mean
    source:                 NOAA CoastWatch West Coast Node
    standard_name:          net_primary_productivity
    units:                  mg C m-2 d-1
NOAA20 Dataset: <xarray.DataArray 'productivity' (time: 60, latitude: 2160, longitude: 4320)> Size: 2GB
[559872000 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 480B 2018-01-16T12:00:00 ... 2022-12-16T...
  * 

In [13]:
modis_ds["time"] = modis_ds["time"] + np.timedelta64(4, "h")
print(modis_ds)

<xarray.DataArray 'productivity' (time: 120, latitude: 2160, longitude: 4320)> Size: 4GB
[1119744000 values with dtype=float32]
Coordinates:
  * latitude   (latitude) float64 17kB 89.96 89.88 89.79 ... -89.88 -89.96
  * longitude  (longitude) float64 35kB -180.0 -179.9 -179.8 ... 179.9 180.0
  * time       (time) datetime64[ns] 960B 2013-01-16T12:00:00 ... 2022-12-16T...
Attributes:
    colorBarMaximum:        30.0
    colorBarMinimum:        0.0
    coverage_content_type:  modelResult
    ioos_category:          Other
    long_name:              Net Primary Productivity Monthly Mean
    source:                 NOAA CoastWatch West Coast Node
    standard_name:          net_primary_productivity
    units:                  mg C m-2 d-1


In [14]:
# Convert coordinates to the same precision
modis_ds["latitude"] = modis_ds["latitude"].astype("float32")
modis_ds["longitude"] = modis_ds["longitude"].astype("float32")
print(modis_ds)

noaa20_ds["latitude"] = noaa20_ds["latitude"].astype("float32")
noaa20_ds["longitude"] = noaa20_ds["longitude"].astype("float32")
print(noaa20_ds)

<xarray.DataArray 'productivity' (time: 120, latitude: 2160, longitude: 4320)> Size: 4GB
[1119744000 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 960B 2013-01-16T12:00:00 ... 2022-12-16T...
  * latitude   (latitude) float32 9kB 89.96 89.88 89.79 ... -89.79 -89.88 -89.96
  * longitude  (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.9 180.0
Attributes:
    colorBarMaximum:        30.0
    colorBarMinimum:        0.0
    coverage_content_type:  modelResult
    ioos_category:          Other
    long_name:              Net Primary Productivity Monthly Mean
    source:                 NOAA CoastWatch West Coast Node
    standard_name:          net_primary_productivity
    units:                  mg C m-2 d-1
<xarray.DataArray 'productivity' (time: 60, latitude: 2160, longitude: 4320)> Size: 2GB
[559872000 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 480B 2018-01-16T12:00:00 ... 2022-12-16T...
  * latitude   (latitude) flo

In [15]:
ds_list = []

# Pre-fetched datasets
datasets = [modis_ds, noaa20_ds]

for ds in datasets:
    ds_dates_list = []  # To store individual monthly slices
    for i in range(ntM):
        # Select a single time slice
        start_date = f"{yy[i]}-{mm[i]:02d}-16"
        try:
            time_slice = ds.sel(time=start_date)
            ds_dates_list.append(time_slice)
        except KeyError as e:
            print(f"Time {start_date} not found in dataset: {e}")
            continue

    # Append the time series for the dataset
    ds_list.append(ds_dates_list)

print(f"Loaded data for {len(ds_list)} datasets.")

Loaded data for 2 datasets.


In [16]:
print(f"Number of datasets: {len(ds_list)}")
print(f"MODIS slices: {len(ds_list[0])}, NOAA20 slices: {len(ds_list[1])}")
print(f"First MODIS slice: {ds_list[0][0]}")
print(f"First NOAA20 slice: {ds_list[1][0]}")

Number of datasets: 2
MODIS slices: 60, NOAA20 slices: 60
First MODIS slice: <xarray.DataArray 'productivity' (time: 1, latitude: 2160, longitude: 4320)> Size: 37MB
[9331200 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 8B 2018-01-16T12:00:00
  * latitude   (latitude) float32 9kB 89.96 89.88 89.79 ... -89.79 -89.88 -89.96
  * longitude  (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.9 180.0
Attributes:
    colorBarMaximum:        30.0
    colorBarMinimum:        0.0
    coverage_content_type:  modelResult
    ioos_category:          Other
    long_name:              Net Primary Productivity Monthly Mean
    source:                 NOAA CoastWatch West Coast Node
    standard_name:          net_primary_productivity
    units:                  mg C m-2 d-1
First NOAA20 slice: <xarray.DataArray 'productivity' (time: 1, latitude: 2160, longitude: 4320)> Size: 37MB
[9331200 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 8B 

In [17]:
# Intialize matrices for correlations and p-values
_, ny1, nx1 = ds_list[0][0].shape
nt1 = len(ds_list[0])

print(f"Temporal Dimension (nt1): {nt1}")
print(f"Latitude Dimension (ny1): {ny1}, Longitude dimensions (nx1): {nx1}")

Temporal Dimension (nt1): 60
Latitude Dimension (ny1): 2160, Longitude dimensions (nx1): 4320


In [18]:
corr_mtrx = np.zeros([ny1, nx1])*np.nan
pval_mtrx = np.zeros([ny1, nx1])*np.nan
n_mtrx = np.zeros([ny1, nx1])*np.nan

In [19]:
# Reshape data into latitude blocks
indx_block = reshape_data_block(
    data=ds_dates_list[0].values,
    block_size=ny_block
)
num_block = indx_block.shape[0]

print(f"Latitude divided into {num_block} blocks of size {ny_block}")

Latitude divided into 108 blocks of size 20


In [20]:
for i in range(num_block):
    print(f"Processing block {i + 1}/{num_block}")
    # Construct the two data matrix of shape [ntM X ny_block X nx1]
    injk = indx_block[i, :]
    data_block_mtrx = np.zeros([len(ds_list), ntM, ny_block, nx1])

    # Populate data_block_mtrx with data from each source and time step
    for j in range(ntM):
        for k in range(len(ds_list)):
            data_block_mtrx[k, j, :, :] = ds_list[k][j].values[0, injk, :]

    # Compute correlations for each latitude subset in the block
    for j in range(ny_block):
        data_mtrx = data_block_mtrx[:, :, j, :]

        # Only keep grid points that have non-missing numbers above 'percent_keep'
        ones_mtrx = data_mtrx/data_mtrx
        ones_mtrx_comb = np.sum(ones_mtrx, axis=0)/2
        sum_one_mtrx = np.nansum(ones_mtrx_comb, axis=0)
        in_keep = np.where(sum_one_mtrx > ntM*percent_keep)[0]

        # Create data array with time as coordinate, useful for
        # Calculating anoms in xarray
        time_series_data = xr.DataArray(
            data_mtrx[:, :, in_keep],
            coords=[
                source_list,
                dtM.astype('datetime64[ns]'),
                in_keep
            ],
            dims=['source', 'time', 'in_keep'])

        # Correlations for either data or anom time series
        if timeseries_type == 'anom':
            # Calculate monthly climatology
            monthly_climatology = time_series_data.groupby('time.month').mean('time')
            # Get anomalies by subtracting climatology from the data
            compare_data = time_series_data.groupby('time.month') - monthly_climatology
        elif timeseries_type == 'data':
            # Use raw data directly for correlation
            compare_data = time_series_data

        # Calculate Pearson correlation between sources
        correlation = xr.corr(
            compare_data.sel(source=source_list[0]),
            compare_data.sel(source=source_list[1]),
            dim='time'
        )

        # Use xskillscore to get p values
        p_value = xs.pearson_r_p_value(
            compare_data.sel(source=source_list[0]),
            compare_data.sel(source=source_list[1]),
            dim='time',
            skipna=True
        )

        # Place corr, pval values for latitude subset in final global data matrix
        corr_mtrx[injk[j], in_keep] = correlation.data
        pval_mtrx[injk[j], in_keep] = p_value.data
        # Store the count of valid data points
        n_mtrx[injk[j], in_keep] = sum_one_mtrx[in_keep]

print("Correlations and p-values computed.")
print(f"Min: {np.nanmin(corr_mtrx)}, Max: {np.nanmax(corr_mtrx)}, Mean: {np.nanmean(corr_mtrx)}")
print(f"Min: {np.nanmin(pval_mtrx)}, Max: {np.nanmax(pval_mtrx)}, Mean: {np.nanmean(pval_mtrx)}")
print(f"Min: {np.nanmin(n_mtrx)}, Max: {np.nanmax(n_mtrx)}, Mean: {np.nanmean(n_mtrx)}")

Processing block 1/108
Processing block 2/108
Processing block 3/108
Processing block 4/108
Processing block 5/108
Processing block 6/108
Processing block 7/108
Processing block 8/108
Processing block 9/108
Processing block 10/108
Processing block 11/108
Processing block 12/108
Processing block 13/108
Processing block 14/108
Processing block 15/108
Processing block 16/108
Processing block 17/108
Processing block 18/108
Processing block 19/108
Processing block 20/108
Processing block 21/108
Processing block 22/108
Processing block 23/108
Processing block 24/108
Processing block 25/108
Processing block 26/108
Processing block 27/108
Processing block 28/108
Processing block 29/108
Processing block 30/108
Processing block 31/108
Processing block 32/108
Processing block 33/108
Processing block 34/108
Processing block 35/108
Processing block 36/108
Processing block 37/108
Processing block 38/108
Processing block 39/108
Processing block 40/108
Processing block 41/108
Processing block 42/108
P

In [21]:
# Mask invalid values in correlation, p-value,
# and count matrices (not sure if need)
corr_mtrx = ma.masked_invalid(corr_mtrx)
pval_mtrx = ma.masked_invalid(pval_mtrx)
n_mtrx = ma.masked_invalid(n_mtrx)

In [22]:
# Labels and matrices to save in NetCDF file
cpn_lbl = ['corr', 'pval', 'n']
cpn_list = [corr_mtrx, pval_mtrx, n_mtrx]

# Open temporary file and load data into it
with netCDF4.Dataset(os.path.join(WORK_DIR, ncgen_ofile_nc), 'a') as nc:
    # Set global attributes
    nc.title = (
        f"Pearson correlation coefficients and p-values of "
        f"monthly primary productivity fields between "
        f"{source1.upper()} and {source2.upper()}"
        )
    nc.summary = (
        f"Correlations between primary productivity or "
        f"PAR, chlorophyl and SST from {source1.upper()} "
        f"and {source2.upper()}. These are 9km products "
        f"generated from time series of monthly means. "
        f"See Melin et al 2017 for more details")
    nc.source = f"{source1.upper()} and {source2.upper()}"
    nc.instrument = f"{source1.upper()} and {source2.upper()}"
    nc.id = f"correlations_{source1}_{source2}_monthly_9km"
    nc.platform = f"{source1.upper()} and {source2.upper()}"
    
    # Place corr, pval, n in nc
    for j in range(len(cpn_lbl)):
        nc['{}'.format(cpn_lbl[j])][0, :, :] = cpn_list[j]

# Compress and save the temporary file to the final file name
myCmd = ' '.join(['nccopy',
                    '-d6',
                    os.path.join(WORK_DIR, ncgen_ofile_nc),
                    os.path.join(odir, ofile)
                    ])
print('nccopy', subprocess.call(myCmd, shell=True))
print('Done with', ofile)

# Clean up temporary file after saving the final output
os.remove(os.path.join(WORK_DIR, ncgen_ofile_nc))

/var/folders/81/qj7mv_yn7p98wpb9n0np6q8c0000gn/T/ipykernel_25493/2503479140.py:26: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  nc['{}'.format(cpn_lbl[j])][0, :, :] = cpn_list[j]


nccopy 0
Done with productivity_data_corr_month_modis_noaa20_9km_201801_202212_050percent.nc
